# 10 · Why paged memory: reserving the maximum wastes the GPU
Level L3 · Part 2 · core · ~15 min · builds on 05

## Goal
By the end of this lesson you can show, with numbers from a made-up Brightside trace, how reserving the longest possible answer for every request wastes KV memory. You also see how many more requests fit when memory is handed out in small blocks.

## Why it matters
- **Without it:** an engine that does not know how long an answer will be reserves room for the longest one allowed. With a 4,096-token limit, every request holds 4,096 token slots, and most stay empty. The vLLM paper reports that in existing systems only **20.4% to 38.2%** of KV cache memory held real token states (arXiv 2309.06180), so the systems it compared left 61.8% to 79.6% empty. That is more waste than this made-up trace shows (about 56%). On one 24 GB L4 with the 8B model (lesson 05), that leaves room for about 9 requests (37,732 ÷ 4,096 = 9.2).
- **With it:** memory is cut into 16-token blocks (vLLM's default), and a request takes blocks as its tokens arrive. The same memory holds more than twice as many requests, and the waste per request is under one block.
- **When you don't need it:** if every request is about the same length, or you serve one user at a time, a fixed reservation wastes little.

## The idea
A hotel that books every guest a full suite for the longest stay the hotel allows, whether they stay one night or a month, fills up fast and sits half empty. Paging is the hotel that lets guests book by the night: a room is handed out when it is needed and returned when the guest leaves. The engine's KV cache works the same way. A request asks for blocks as its answer grows, and gives them back when it finishes.

## Picture
**Without paging:** each request owns one slab sized for the longest answer.
```mermaid
flowchart LR
  R1["request 1: reserves 4,096 slots"] --> M["one contiguous slab per request"]
  R2["request 2: reserves 4,096 slots"] --> M
  M --> U["answer uses about 1,900 slots; the rest stays empty"]
  U --> X["no room left for request 10"]
```

**With paging:** one pool of small blocks, handed out as tokens arrive.
```mermaid
flowchart LR
  P["one pool of 16-token blocks"] --> A["request 1 takes blocks 7, 12, 3 as it grows"]
  P --> B["request 2 takes blocks 1, 9"]
  A --> T["block table: request's token range to physical block"]
  B --> T
  T --> F["request finishes: blocks return to the pool"]
```

## How it simplifies the architecture
| | Without paging | With paging |
|---|---|---|
| Unit of memory | one slab per request, sized for the maximum | 16-token blocks, taken as needed |
| Who tracks memory | a slot counter per request | a pool plus a block table per request |
| Moving parts | one allocator: reserve, release | block pool, block table, free list (lesson 11 builds them) |
| Waste per request | up to the whole reservation | at most one block (15 slots) |
| Failure point | a request longer than the limit is refused | the pool can run out mid-answer; the engine must preempt (lesson 13) |

Paging adds bookkeeping, but it removes the reservation glue and most of the waste. What it does not solve is named at the end.

## Already know this?
This is the first lesson of Part 2 (memory, batching and engines). Answer these six questions. If you can answer all six, skip to lesson 17. Lesson 17 adds speculative decoding and multi-LoRA, which none of the six questions touch. Lesson 18 is the Part checkpoint.
1. What does a KV budget of 37,700 tokens mean, and which lesson computed it? (lesson 05)
2. When a short request finishes inside a running batch, what happens to the other requests? (lesson 12)
3. What must match for a prefix cache hit? (lesson 14)
4. What can an engine do when the KV pool runs out in the middle of an answer? (lesson 13)
5. Which setting caps the context length, and which caps the number of requests run at once? (lesson 16)
6. Name two engines besides vLLM and one reason to pick one. (lesson 15)

## Step by step
### Step 1: a made-up support-desk trace
Every Brightside request sends the same 1,500-token policy prompt, then the customer's question, then the answer. We draw 200 requests with a fixed seed. The lengths are made up, but the shape matches a support desk: most answers are short, a few are long. The limit is 4,096 tokens for prompt plus answer.

In [1]:
import random

POLICY_TOKENS = 1_500   # the shared Brightside policy prompt (made-up trace; lengths are invented)
MAX_LEN = 4_096         # the context limit the team sets for every request

def make_trace(n=200, seed=7):
    """A made-up trace: question and answer lengths from a seeded draw.
    Note: this reads POLICY_TOKENS when it is called, so change the global before calling it."""
    rng = random.Random(seed)
    rows = []
    for i in range(n):
        question = rng.randint(30, 200)
        answer = rng.randint(50, 250) if rng.random() < 0.8 else rng.randint(400, 1_200)
        total = min(POLICY_TOKENS + question + answer, MAX_LEN)   # the server stops at the limit
        rows.append({"id": i, "total": total})
    return rows

trace = make_trace()
print(f"{len(trace)} requests, average total {sum(r['total'] for r in trace) / len(trace):,.0f} tokens")
for row in trace[:5]:
    print(row)

200 requests, average total 1,891 tokens
{'id': 0, 'total': 2416}
{'id': 1, 'total': 1883}
{'id': 2, 'total': 1618}
{'id': 3, 'total': 1731}
{'id': 4, 'total': 1752}


### Step 2: the KV budget from lesson 05
On one 24 GB L4 with the 8B model in FP16, lesson 05 gives about 37,700 tokens of KV cache (lesson 05 got 37,732 too). The numbers are restated here so the lesson runs on its own. The 1 GiB activation figure is assumed, as in lesson 05.

In [2]:
GIB = 2**30
KV_PER_TOKEN = 131_072           # Llama 3.1 8B, FP16: 128 KiB per token (lesson 03)
GPU = 24e9 * 0.92                # memory the engine may use (gpu_memory_utilization 0.92)
WEIGHTS = 8_030_261_248 * 2      # 8.03 billion parameters x 2 bytes (lesson 05)
ACTIVATIONS = 1 * GIB            # assumed, as in lesson 05
BUDGET_TOKENS = int((GPU - WEIGHTS - ACTIVATIONS) // KV_PER_TOKEN)
print(f"KV budget: {BUDGET_TOKENS:,} tokens")

KV budget: 37,732 tokens


### Step 3: without paging, reserve the maximum
This is the world without paging, written by hand. Each request takes `MAX_LEN` slots when it starts. A request is admitted only if its full reservation fits, and the first request that does not fit stops the queue.
"Empty" means `1 - used / reserved`: the share of the reserved slots that hold no token.

In [3]:
def admit_reserved(budget_tokens, rows, max_len=MAX_LEN):
    """Without paging: each admitted request reserves max_len token slots up front."""
    admitted, free = [], budget_tokens
    for row in rows:
        if row["total"] > max_len:
            raise ValueError(f"request {row['id']} needs {row['total']} tokens, above max_len {max_len}")
        if free < max_len:
            break                       # first come, first served: stop at the first that does not fit
        free -= max_len
        admitted.append(row)
    return admitted

def report(name, admitted, reserved_tokens):
    """Requests admitted, tokens really used, tokens reserved, and the share left empty (1 - used / reserved)."""
    used = sum(r["total"] for r in admitted)
    waste = 1 - used / reserved_tokens
    print(f"{name:10} {len(admitted):3} requests  used {used:7,}  reserved {reserved_tokens:7,}  empty {waste:.1%}")

naive = admit_reserved(BUDGET_TOKENS, trace)
report("reserve", naive, len(naive) * MAX_LEN)

reserve      9 requests  used  16,294  reserved  36,864  empty 55.8%


**What you should see:** 9 requests run, and more than half of the reserved slots stay empty (about 56%, from `1 - used / reserved`). The requests use fewer tokens than their reservation, and the reservation is the only thing the budget counts. The 9 reservations take 36,864 of the 37,732 tokens, so about 870 tokens of budget stay unused.

### Step 4: with paging, hand out 16-token blocks
Now the same budget is cut into blocks of 16 tokens. vLLM uses 16 as its default block size (vLLM source, `DEFAULT_BLOCK_SIZE`; section 7.2 of the paper). A request needs `ceil(total / 16)` blocks. Rounding up means a partly filled block costs a whole block, so the waste is at most 15 slots per request.

In [4]:
BLOCK = 16

def blocks_for(tokens, block=BLOCK):
    """Blocks a request needs: a partly filled block still costs a whole block."""
    return -(-tokens // block)          # ceiling division

def admit_paged(budget_tokens, rows, block=BLOCK):
    """With paging: take the blocks each request needs, first come, first served."""
    free, admitted = budget_tokens // block, []
    for row in rows:
        need = blocks_for(row["total"], block)
        if need > free:
            break
        free -= need
        admitted.append(row)
    return admitted

paged = admit_paged(BUDGET_TOKENS, trace)
report("paged", paged, sum(blocks_for(r["total"]) for r in paged) * BLOCK)

paged       21 requests  used  37,482  reserved  37,648  empty 0.4%


The paged "empty" is not zero, only small: the real figure is 0.4% (166 of 37,648 reserved slots). Those slots are the partly filled last block of each request, which is the one-block waste from the Picture. The loop stops at 21 requests: the next one needs more blocks than the 5 left, so about 80 tokens of the budget stay unused.

### Step 5: the same budget, side by side
The only change is how memory is handed out. The budget and the trace are the same.

In [5]:
print(f"requests that fit: reserve {len(naive)}, paged {len(paged)} "
      f"({len(paged) / len(naive):.1f}x) on the same {BUDGET_TOKENS:,}-token budget")

requests that fit: reserve 9, paged 21 (2.3x) on the same 37,732-token budget


**What you should see:** paging admits more than twice as many requests (21 against 9). The gain comes from the empty slots, not from a faster GPU. This is a snapshot: it counts each request's final length. In a live engine blocks are taken as the answer grows, and lesson 13 covers what happens when the pool runs out.

## At work
- **Meta** splits prefill and decode across hosts. Its PyTorch write-up (2025-09-12) says vLLM suggests 16-token blocks, but many small blocks meant many small kernel launches when moving KV between CPU and GPU, so Meta uses larger ones (e.g. 128 or 256). → lesson 34.
- **vLLM** prints the KV cache size and the maximum concurrency in its startup log. Compare that number with the one you calculated here. → lesson 16.
- **Databricks** reports that continuous batching gives 10 to 20 times the throughput of dynamic batching; the batch size you can run is set by KV memory. → lesson 12.

Sources: vLLM paper, arXiv 2309.06180 (Introduction; section 7.2) · vLLM `vllm/config/cache.py` (`DEFAULT_BLOCK_SIZE = 16`): https://github.com/vllm-project/vllm · PyTorch blog: https://pytorch.org/blog/disaggregated-inference-at-scale-with-pytorch-vllm/ · Databricks: https://www.databricks.com/blog/llm-inference-performance-engineering-best-practices

## Common mistakes
**1. Raising the limit "just in case".** A higher `MAX_LEN` protects long answers, but every request now reserves more. With the cap at 8,192 the naive reservation admits 4 requests instead of 9.

In [6]:
naive_big = admit_reserved(BUDGET_TOKENS, trace, max_len=8_192)
print(f"reserve with an 8,192 limit: {len(naive_big)} requests (was {len(naive)})")

reserve with an 8,192 limit: 4 requests (was 9)


**2. Sending a request above the limit.** The reserving allocator refuses it. The error is the signal: either cap the answer length (`max_tokens`) or raise the limit and pay for it in memory. Here one 5,000-token request arrives.

In [7]:
admit_reserved(BUDGET_TOKENS, [{"id": 999, "total": 5_000}])   # ValueError: above max_len

ValueError: request 999 needs 5000 tokens, above max_len 4096

**3. Rounding down the blocks.** `tokens // 16` leaves out the last partly filled block. A 1,801-token request needs 113 blocks, not 112.

In [8]:
print(f"1,801 tokens: floor {1801 // 16} blocks (wrong), ceiling {blocks_for(1801)} blocks (right)")

1,801 tokens: floor 112 blocks (wrong), ceiling 113 blocks (right)


## Try it
Brightside adds a longer policy: the prompt grows to 2,000 tokens, so every request is about 500 tokens longer. Rebuild the trace with `POLICY_TOKENS = 2_000` and count how many requests fit with the reservation and with paging on the same budget. The trace has a cap, so keep `MAX_LEN` at 4,096.

In [9]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [10]:
POLICY_TOKENS = 2_000
longer = make_trace()
print(f"policy 2,000 tokens: reserve {len(admit_reserved(BUDGET_TOKENS, longer))}, "
      f"paged {len(admit_paged(BUDGET_TOKENS, longer))} requests")
POLICY_TOKENS = 1_500

policy 2,000 tokens: reserve 9, paged 16 requests


The reservation still admits 9 requests because it ignores the real length: every request costs `MAX_LEN` whatever its prompt holds. Paging feels the longer prompt, since each request now takes more blocks, so its count drops from 21 to 16.

## What paging does not fix, and what else you could do
Paging is not the only fix. Cheaper options come first:
- **Cap lengths per route.** Set `max_tokens` and a lower `MAX_LEN` where answers are short. A yes/no route does not need 4,096 slots.
- **Reserve by length bucket.** A short pool and a long pool stop a 300-token answer from reserving the longest slab.
- **Reserve for the predicted answer length,** from the route's history, plus a margin.
- **Keep the fixed reservation** for one user or uniform-length traffic, where it wastes little.

Still open after paging: pool exhaustion mid-answer (lesson 13), prefix reuse (lesson 14), block size across nodes (lesson 34).

## Recap
- Reserving the maximum length for each request left 55.8% of the reserved KV memory empty in this trace.
- Paging hands out 16-token blocks as tokens arrive, so the waste is at most one block per request.
- The same 37,700-token budget admits 21 requests with paging against 9 with reservation. The numbers come from one made-up trace.
- Paging does not solve pool exhaustion, prefix reuse or the block size. Lessons 13, 14 and 34 do.

**Interview:** "Why can a vLLM-style engine serve more requests than a server that reserves the maximum length per request, on the same GPU?" Strong answer: a fixed reservation sized for the maximum wastes most of its slots; a block table hands out small blocks as tokens arrive, so the budget holds more live requests, and the waste is bounded by one block per request.